# Regenerate All Manuscript Figures -- Bioinformatics Submission

This notebook regenerates **all 10 manuscript figures** at larger sizes (400 DPI) for the Bioinformatics submission.

**Original figures (1--6):**
1. Distribution of resistant/susceptible sequences by drug
2. Statistical comparison of ESM-2 vs baseline methods
3. Classifier architecture comparison on ESM-2 embeddings
4. DRM enrichment validation
5. Attention differential profiles for representative drugs
6. Calibration curves (9-panel grid)

**New figures (7--10):**
7. Multi-PLM comparison (bar chart + box plot)
8. PLM comparison heatmap
9. Subtype-stratified performance
10. Temporal holdout AUC

**Runtime:** ~25--35 minutes on Colab T4 GPU

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
!pip install -q esm torch transformers xgboost scikit-learn seaborn openpyxl biopython tqdm

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Clone repo and upload data ──────────────────────────────────────
!git clone https://github.com/hayden-farquhar/HIV-ESM-2.git

import sys, os
os.chdir('HIV-ESM-2')
sys.path.insert(0, '.')

# Create directories
os.makedirs('data/embeddings', exist_ok=True)
os.makedirs('results/robustness', exist_ok=True)
os.makedirs('figures', exist_ok=True)
os.makedirs('figures/robustness', exist_ok=True)

In [ ]:
# ── Cell 3: Upload the 3 HIVDB data files ──────────────────────────────────
from google.colab import files
print('Upload PI_dataset.txt, NRTI_dataset.txt, NNRTI_dataset.txt')
uploaded = files.upload()

os.makedirs('data_raw', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'data_raw/{fname}')
    print(f'  Moved {fname} -> data_raw/{fname}')

from pathlib import Path
DATA_DIR = Path('data_raw')

In [ ]:
# ── Cell 4: Imports and global settings ────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from src.data_processing import PI_DRUGS, NRTI_DRUGS, NNRTI_DRUGS, get_drug_class
from src.feature_engineering import (
    create_binary_mutation_encoding,
    HIV_PROTEASE_REFERENCE,
    HIV_RT_REFERENCE,
)
from src.models import per_drug_training, compare_models, train_attention_model
from src.evaluation import (
    platt_scaling, isotonic_calibration,
    compute_calibration_metrics, compare_esm2_vs_baseline,
)
from src.visualization import (
    plot_attention_heatmap, plot_drm_enrichment, plot_calibration_curve,
)
from src.interpretability import (
    load_known_drms, get_drug_specific_drms,
    compute_drm_enrichment, compute_learned_attention_differential,
)
from src.subtype_analysis import (
    reconstruct_all_datasets,
    assign_subtypes_via_sequence_similarity,
    subtype_stratified_evaluation,
    create_temporal_split,
    temporal_holdout_evaluation,
)

# ── Updated visualization defaults ──
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 400
plt.rcParams['font.size'] = 13

FIGURES_DIR = Path('figures')
RESULTS_DIR = Path('results/robustness')
EMBEDDING_DIR = Path('data/embeddings')

drug_lists = {'PI': PI_DRUGS, 'NRTI': NRTI_DRUGS, 'NNRTI': NNRTI_DRUGS}

---
## Step 1: Reconstruct sequences and binarize phenotypes

In [ ]:
# ── Cell 5: Reconstruct sequences ─────────────────────────────────────────
print('=== Reconstructing sequences ===')
datasets = reconstruct_all_datasets(DATA_DIR)
for dc, df in datasets.items():
    print(f'  {dc}: {len(df)} sequences, length={df["sequence"].str.len().iloc[0]}')

In [ ]:
# ── Cell 6: Binarize phenotypes ───────────────────────────────────────────
pheno_dfs = {}
for dc, df in datasets.items():
    pheno = pd.DataFrame()
    for drug in drug_lists[dc]:
        if drug in df.columns:
            fc = pd.to_numeric(df[drug], errors='coerce')
            pheno[drug] = np.where(fc.isna(), np.nan, (fc >= 2.5).astype(float))
    pheno_dfs[dc] = pheno
    n_valid = pheno.notna().sum()
    print(f'{dc}: {len(pheno)} samples')
    for drug in drug_lists[dc]:
        if drug in pheno.columns:
            y = pheno[drug].dropna()
            print(f'  {drug}: n={len(y)}, R={int(y.sum())}, S={int(len(y)-y.sum())}')

---
## Step 2: Extract ESM-2 embeddings (mean-pooled + per-residue)

In [ ]:
# ── Cell 7: Load ESM-2 and extract embeddings ─────────────────────────────
from transformers import EsmModel, EsmTokenizer

print('Loading ESM-2 650M...')
tokenizer = EsmTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
esm2_model = EsmModel.from_pretrained('facebook/esm2_t33_650M_UR50D').to(device).eval()
print(f'ESM-2 loaded on {device}')


def extract_embeddings_hf(sequences, model, tokenizer, device, batch_size=16):
    """Extract mean-pooled embeddings via HuggingFace transformers."""
    all_pooled = []
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = sequences[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=1024)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        for j, seq in enumerate(batch):
            emb = out.last_hidden_state[j, 1:len(seq)+1, :].cpu().numpy()
            all_pooled.append(emb.mean(axis=0))
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return np.array(all_pooled)


def extract_per_residue_hf(sequences, model, tokenizer, device, batch_size=16):
    """Extract per-residue embeddings via HuggingFace transformers."""
    all_embs = []
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = sequences[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=1024)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        for j, seq in enumerate(batch):
            emb = out.last_hidden_state[j, 1:len(seq)+1, :].cpu().numpy()
            all_embs.append(emb)
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return all_embs


# Extract mean-pooled embeddings
esm2_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esm2_embeddings.npy'
    if cache.exists():
        esm2_embeddings[dc] = np.load(cache)
        print(f'{dc}: loaded cached {esm2_embeddings[dc].shape}')
    else:
        print(f'{dc}: extracting mean-pooled embeddings...')
        X = extract_embeddings_hf(
            df['sequence'].tolist(), esm2_model, tokenizer, device, batch_size=16
        )
        np.save(cache, X)
        esm2_embeddings[dc] = X
        print(f'  {dc}: {X.shape}')

# Extract per-residue embeddings (needed for attention classifier)
per_residue_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esm2_per_residue.npy'
    if cache.exists():
        per_residue_embeddings[dc] = list(np.load(cache, allow_pickle=True))
        print(f'{dc}: loaded cached per-residue ({len(per_residue_embeddings[dc])} seqs)')
    else:
        print(f'{dc}: extracting per-residue embeddings...')
        embs = extract_per_residue_hf(
            df['sequence'].tolist(), esm2_model, tokenizer, device, batch_size=16
        )
        np.save(cache, np.array(embs, dtype=object), allow_pickle=True)
        per_residue_embeddings[dc] = embs
        print(f'  {dc}: {len(embs)} sequences')

---
## Step 3: Run ESM-2 classifiers and baseline

In [ ]:
# ── Cell 8: ESM-2 logistic regression (primary model) ─────────────────────
print('=== ESM-2 + Logistic Regression (primary) ===')
esm2_lr_results = {}
for dc in ['PI', 'NRTI', 'NNRTI']:
    print(f'\n--- {dc} ---')
    res = per_drug_training(
        esm2_embeddings[dc], pheno_dfs[dc], drug_lists[dc],
        model_type='logistic', n_splits=5, random_state=42
    )
    esm2_lr_results.update(res)

print(f'\nMean ESM-2 AUC: {np.mean([r["auc"] for r in esm2_lr_results.values()]):.4f}')

In [ ]:
# ── Cell 9: Baseline -- XGBoost on binary mutation encoding ───────────────
print('=== Baseline: XGBoost on binary mutation encoding ===')
references = {
    'PI': HIV_PROTEASE_REFERENCE,
    'NRTI': HIV_RT_REFERENCE,
    'NNRTI': HIV_RT_REFERENCE,
}

baseline_results = {}
for dc in ['PI', 'NRTI', 'NNRTI']:
    print(f'\n--- {dc} ---')
    seqs = datasets[dc]['sequence'].tolist()
    X_mut = create_binary_mutation_encoding(seqs, references[dc])
    print(f'  Mutation encoding shape: {X_mut.shape}')
    res = per_drug_training(
        X_mut, pheno_dfs[dc], drug_lists[dc],
        model_type='xgboost', n_splits=5, random_state=42
    )
    baseline_results.update(res)

print(f'\nMean baseline AUC: {np.mean([r["auc"] for r in baseline_results.values()]):.4f}')

In [ ]:
# ── Cell 10: Multi-classifier comparison on ESM-2 embeddings ─────────────
print('=== Classifier comparison on frozen ESM-2 embeddings ===')
classifier_results = {}

for model_type in ['logistic', 'xgboost', 'rf']:
    print(f'\n--- {model_type.upper()} ---')
    classifier_results[model_type] = {}
    for dc in ['PI', 'NRTI', 'NNRTI']:
        res = per_drug_training(
            esm2_embeddings[dc], pheno_dfs[dc], drug_lists[dc],
            model_type=model_type, n_splits=5, random_state=42
        )
        classifier_results[model_type].update(res)

# Summary
for mt, res in classifier_results.items():
    mean_auc = np.mean([r['auc'] for r in res.values()])
    print(f'{mt}: mean AUC = {mean_auc:.4f}')

In [ ]:
# ── Cell 11: Train attention-weighted classifier (for Figures 4-5) ────────
print('=== Training attention-weighted classifiers ===')
attention_models = {}
attention_results = {}

for dc in ['PI', 'NRTI', 'NNRTI']:
    print(f'\n--- {dc} ---')
    # Train one attention model per drug for interpretability
    drugs = drug_lists[dc]
    pheno = pheno_dfs[dc]
    embs = per_residue_embeddings[dc]

    for drug in drugs:
        col = drug
        if col not in pheno.columns:
            continue
        y = pheno[col].values
        valid_mask = ~np.isnan(y)
        embs_valid = [embs[i] for i in range(len(embs)) if valid_mask[i]]
        y_valid = y[valid_mask].astype(int)

        if len(np.unique(y_valid)) < 2 or len(y_valid) < 50:
            continue

        # Train/val split for attention model
        from sklearn.model_selection import train_test_split
        idx = np.arange(len(y_valid))
        train_idx, val_idx = train_test_split(
            idx, test_size=0.2, stratify=y_valid, random_state=42
        )
        train_embs = [embs_valid[i] for i in train_idx]
        val_embs = [embs_valid[i] for i in val_idx]
        y_train = y_valid[train_idx]
        y_val = y_valid[val_idx]

        model = train_attention_model(
            train_embs, y_train,
            val_embeddings_list=val_embs, val_labels=y_val,
            input_dim=1280, attention_dim=256,
            batch_size=32, epochs=20, lr=1e-4,
            device=device, verbose=False
        )
        attention_models[drug] = model
        print(f'  {drug}: trained attention classifier')

print(f'\nTrained {len(attention_models)} attention models')

---
## Step 4: DRM enrichment analysis (for Figure 4)

In [ ]:
# ── Cell 12: Compute DRM enrichment for all drugs ────────────────────────
print('=== DRM enrichment analysis ===')
all_validation_rows = []

for dc in ['PI', 'NRTI', 'NNRTI']:
    print(f'\n--- {dc} ---')
    drugs = drug_lists[dc]
    pheno = pheno_dfs[dc]
    embs = per_residue_embeddings[dc]

    for drug in drugs:
        if drug not in attention_models:
            print(f'  Skipping {drug}: no attention model')
            continue

        col = drug
        if col not in pheno.columns:
            continue
        y = pheno[col].values
        valid_mask = ~np.isnan(y)
        embs_valid = [embs[i] for i in range(len(embs)) if valid_mask[i]]
        y_valid = y[valid_mask].astype(int)

        if len(np.unique(y_valid)) < 2:
            continue

        # Compute learned attention differential
        diff_result = compute_learned_attention_differential(
            embs_valid, y_valid, attention_models[drug], device,
            max_samples=100, random_state=42
        )
        if diff_result is None:
            continue

        differential = diff_result['differential']
        drm_positions = get_drug_specific_drms(drug, dc)

        # Test multiple top-k thresholds
        for top_k in [5, 10, 15, 20, 25, 30]:
            enrichment = compute_drm_enrichment(differential, drm_positions, top_k=top_k)
            enrichment['drug'] = drug
            enrichment['drug_class'] = dc
            all_validation_rows.append(enrichment)

        print(f'  {drug}: enrichment={enrichment["enrichment_ratio"]:.2f}, p={enrichment["p_value"]:.4f}')

validation_df = pd.DataFrame(all_validation_rows)
print(f'\nTotal validation entries: {len(validation_df)}')

---
## Step 5: Attention differential profiles (for Figure 5)

In [ ]:
# ── Cell 13: Compute attention differentials for representative drugs ─────
print('=== Attention differential profiles ===')
representative_drugs = {
    'ATV': 'PI',
    'ABC': 'NRTI',
    'EFV': 'NNRTI',
}

attention_differentials = {}
for drug, dc in representative_drugs.items():
    if drug not in attention_models:
        print(f'  Skipping {drug}: no attention model')
        continue

    pheno = pheno_dfs[dc]
    embs = per_residue_embeddings[dc]
    col = drug
    y = pheno[col].values
    valid_mask = ~np.isnan(y)
    embs_valid = [embs[i] for i in range(len(embs)) if valid_mask[i]]
    y_valid = y[valid_mask].astype(int)

    diff_result = compute_learned_attention_differential(
        embs_valid, y_valid, attention_models[drug], device,
        max_samples=100, random_state=42
    )
    if diff_result is not None:
        attention_differentials[drug] = diff_result
        print(f'  {drug}: computed differential (len={len(diff_result["differential"])})')

---
## Step 6: Calibration analysis (for Figure 6)

In [ ]:
# ── Cell 14: Compute calibration data for all drug classes ────────────────
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print('=== Calibration analysis ===')
calibration_data = {}  # (drug_class, method) -> (y_true, y_pred)

for dc in ['PI', 'NRTI', 'NNRTI']:
    print(f'\n--- {dc} ---')
    X = esm2_embeddings[dc]
    drugs = drug_lists[dc]

    # Aggregate predictions across all drugs in this class
    all_y_true = []
    all_y_raw = []
    all_y_platt = []
    all_y_isotonic = []

    for drug in drugs:
        col = drug
        if col not in pheno_dfs[dc].columns:
            continue
        y = pheno_dfs[dc][col].values
        valid_mask = ~np.isnan(y)
        X_valid = X[valid_mask]
        y_valid = y[valid_mask].astype(int)

        if len(np.unique(y_valid)) < 2 or len(y_valid) < 50:
            continue

        # Split: 60% train, 20% calibration, 20% test
        X_tr, X_rem, y_tr, y_rem = train_test_split(
            X_valid, y_valid, test_size=0.4, stratify=y_valid, random_state=42
        )
        X_cal, X_test, y_cal, y_test = train_test_split(
            X_rem, y_rem, test_size=0.5, stratify=y_rem, random_state=42
        )

        # Train logistic regression
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_cal_s = scaler.transform(X_cal)
        X_test_s = scaler.transform(X_test)

        lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
        lr.fit(X_tr_s, y_tr)

        y_raw_cal = lr.predict_proba(X_cal_s)[:, 1]
        y_raw_test = lr.predict_proba(X_test_s)[:, 1]

        # Platt scaling
        y_platt_test = platt_scaling(y_cal, y_raw_cal, y_raw_test)
        # Isotonic regression
        y_iso_test = isotonic_calibration(y_cal, y_raw_cal, y_raw_test)

        all_y_true.extend(y_test)
        all_y_raw.extend(y_raw_test)
        all_y_platt.extend(y_platt_test)
        all_y_isotonic.extend(y_iso_test)

    calibration_data[dc] = {
        'y_true': np.array(all_y_true),
        'raw': np.array(all_y_raw),
        'platt': np.array(all_y_platt),
        'isotonic': np.array(all_y_isotonic),
    }
    print(f'  {dc}: {len(all_y_true)} test samples aggregated')

---
## Step 7: Robustness analyses (for Figures 7--10)

In [ ]:
# ── Cell 15: Subtype assignment ──────────────────────────────────────────
print('=== Assigning subtypes ===')
subtype_dfs = {}
for dc, df in datasets.items():
    st_df = assign_subtypes_via_sequence_similarity(
        df['sequence'].tolist(), df['SeqID'].tolist(), drug_class=dc
    )
    subtype_dfs[dc] = st_df
    print(f'\n{dc}:')
    print(st_df['subtype'].value_counts())

In [ ]:
# ── Cell 16: Subtype-stratified evaluation ───────────────────────────────
print('=== Subtype-stratified AUC ===')
all_strat = []
for dc in datasets:
    print(f'\n--- {dc} ---')
    strat = subtype_stratified_evaluation(
        esm2_embeddings[dc], pheno_dfs[dc], subtype_dfs[dc]['subtype'],
        drug_lists[dc], model_type='logistic'
    )
    strat['drug_class'] = dc
    all_strat.append(strat)

strat_df = pd.concat(all_strat, ignore_index=True)
strat_df.to_csv(RESULTS_DIR / 'subtype_stratified_auc.csv', index=False)
print('\nMean AUC by subtype:')
print(strat_df.groupby('subtype')['auc'].agg(['mean', 'std', 'count']))

In [ ]:
# ── Cell 17: Temporal holdout ────────────────────────────────────────────
print('=== Temporal holdout ===')
all_temp = []
for dc in datasets:
    print(f'\n--- {dc} ---')
    train_idx, test_idx = create_temporal_split(datasets[dc], cutoff_quantile=0.8)
    temp = temporal_holdout_evaluation(
        esm2_embeddings[dc], pheno_dfs[dc], drug_lists[dc],
        train_idx, test_idx, model_type='logistic'
    )
    temp['drug_class'] = dc
    all_temp.append(temp)

temp_df = pd.concat(all_temp, ignore_index=True)
temp_df.to_csv(RESULTS_DIR / 'temporal_holdout_auc.csv', index=False)
print(f'\nMean temporal holdout AUC: {temp_df["auc"].mean():.4f}')

In [ ]:
# ── Cell 18: Multi-PLM comparison ────────────────────────────────────────
# ESM-2 already extracted; now ESM C and ESM-1v

# ESM C 600M
from esm.models.esmc import ESMC
from esm.tokenization import EsmSequenceTokenizer

print('Loading ESM C 600M...')
esmc_model = ESMC.from_pretrained('esmc_600m', device=device)
esmc_model.eval()
esmc_tokenizer = EsmSequenceTokenizer()
print(f'ESM C loaded on {device}')


def extract_esmc_embs(sequences, model, tokenizer, device, batch_size=16):
    all_pooled = []
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = sequences[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, return_tensors='pt')
        input_ids = tokens['input_ids'].to(device)
        with torch.no_grad():
            out = model(input_ids)
        for j, seq in enumerate(batch):
            emb = out.embeddings[j, 1:len(seq)+1, :].cpu().float().numpy()
            all_pooled.append(emb.mean(axis=0))
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return np.array(all_pooled)


esmc_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esmc_embeddings.npy'
    if cache.exists():
        esmc_embeddings[dc] = np.load(cache)
        print(f'{dc}: loaded cached {esmc_embeddings[dc].shape}')
    else:
        print(f'{dc}: extracting ESM C embeddings...')
        X = extract_esmc_embs(df['sequence'].tolist(), esmc_model, esmc_tokenizer, device)
        np.save(cache, X)
        esmc_embeddings[dc] = X
        print(f'  {dc}: {X.shape}')

del esmc_model
torch.cuda.empty_cache() if device.type == 'cuda' else None

# ESM-1v
print('\nLoading ESM-1v...')
esm1v_tokenizer = EsmTokenizer.from_pretrained('facebook/esm1v_t33_650M_UR90S_1')
esm1v_model = EsmModel.from_pretrained('facebook/esm1v_t33_650M_UR90S_1').to(device).eval()
print(f'ESM-1v loaded on {device}')

esm1v_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esm1v_embeddings.npy'
    if cache.exists():
        esm1v_embeddings[dc] = np.load(cache)
        print(f'{dc}: loaded cached {esm1v_embeddings[dc].shape}')
    else:
        print(f'{dc}: extracting ESM-1v embeddings...')
        X = extract_embeddings_hf(
            df['sequence'].tolist(), esm1v_model, esm1v_tokenizer, device, batch_size=16
        )
        np.save(cache, X)
        esm1v_embeddings[dc] = X
        print(f'  {dc}: {X.shape}')

del esm1v_model
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# ── Cell 19: Run PLM comparison classifiers ──────────────────────────────
all_plm_results = []

for plm_name, emb_dict in [('ESM-2', esm2_embeddings),
                            ('ESM-C', esmc_embeddings),
                            ('ESM-1v', esm1v_embeddings)]:
    print(f'\n=== {plm_name} ===')
    for dc in ['PI', 'NRTI', 'NNRTI']:
        X = emb_dict[dc]
        drugs = drug_lists[dc]
        pheno = pheno_dfs[dc]
        print(f'\n  {dc} ({X.shape[1]}-dim):')
        results = per_drug_training(
            X, pheno, drugs, model_type='logistic', n_splits=5, random_state=42
        )
        for drug, res in results.items():
            all_plm_results.append({
                'plm': plm_name, 'drug_class': dc, 'drug': drug,
                'auc': res['auc'], 'n_samples': res['n_samples'],
                'embed_dim': X.shape[1]
            })

plm_df = pd.DataFrame(all_plm_results)
plm_df.to_csv(RESULTS_DIR / 'plm_comparison_results.csv', index=False)

print('\n=== Mean AUC by PLM ===')
print(plm_df.groupby('plm')['auc'].agg(['mean', 'std']).round(4))

---
## Figure Generation

All intermediate results are now computed. Generate all 10 figures below.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Distribution of resistant/susceptible sequences by drug
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 1: Dataset composition ===')

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for idx, (dc, ax) in enumerate(zip(['PI', 'NRTI', 'NNRTI'], axes)):
    drugs = drug_lists[dc]
    pheno = pheno_dfs[dc]
    n_resistant = []
    n_susceptible = []
    valid_drugs = []

    for drug in drugs:
        if drug not in pheno.columns:
            continue
        y = pheno[drug].dropna()
        n_r = int(y.sum())
        n_s = int(len(y) - y.sum())
        n_resistant.append(n_r)
        n_susceptible.append(n_s)
        valid_drugs.append(drug)

    x = np.arange(len(valid_drugs))
    width = 0.4

    ax.bar(x - width/2, n_susceptible, width, label='Susceptible', color='#3498db', alpha=0.85)
    ax.bar(x + width/2, n_resistant, width, label='Resistant', color='#e74c3c', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(valid_drugs, rotation=45, ha='right')
    ax.set_ylabel('Number of Sequences', fontsize=13)
    ax.set_title(f'{dc}', fontsize=14, fontweight='bold')
    ax.legend()

    # Add count labels
    for i, (ns, nr) in enumerate(zip(n_susceptible, n_resistant)):
        ax.text(i - width/2, ns + 20, str(ns), ha='center', fontsize=7, rotation=90)
        ax.text(i + width/2, nr + 20, str(nr), ha='center', fontsize=7, rotation=90)

fig.suptitle('Figure 1: Distribution of Resistant and Susceptible Sequences by Drug',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure1_dataset_composition.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure1_dataset_composition.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure1_dataset_composition.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Statistical comparison of methods
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 2: ESM-2 vs Baseline comparison ===')

fig, axes = plt.subplots(1, 3, figsize=(22, 8))

# ── Panel A: Mean AUC by method (horizontal bars) ──
ax = axes[0]
common_drugs = sorted(set(esm2_lr_results.keys()) & set(baseline_results.keys()))

esm2_aucs = [esm2_lr_results[d]['auc'] for d in common_drugs]
baseline_aucs = [baseline_results[d]['auc'] for d in common_drugs]

methods = ['XGBoost\n(Binary Mutations)', 'Logistic Regression\n(ESM-2 Embeddings)']
means = [np.mean(baseline_aucs), np.mean(esm2_aucs)]
stds = [np.std(baseline_aucs), np.std(esm2_aucs)]
colors = ['#3498db', '#e74c3c']

bars = ax.barh(methods, means, xerr=stds, color=colors, alpha=0.85,
               edgecolor='white', capsize=5)
ax.set_xlabel('Mean AUC-ROC', fontsize=13)
ax.set_title('A) Mean AUC by Method', fontsize=14, fontweight='bold')
ax.set_xlim([0.8, 1.0])
for bar, m in zip(bars, means):
    ax.text(m + 0.005, bar.get_y() + bar.get_height()/2,
            f'{m:.4f}', va='center', fontsize=11, fontweight='bold')

# ── Panel B: Per-drug AUC comparison (scatter) ──
ax = axes[1]
drug_classes_for_color = [get_drug_class(d) for d in common_drugs]
color_map = {'PI': '#2196F3', 'NRTI': '#4CAF50', 'NNRTI': '#FF9800'}
scatter_colors = [color_map[dc] for dc in drug_classes_for_color]

ax.scatter(baseline_aucs, esm2_aucs, c=scatter_colors, s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.plot([0.8, 1.0], [0.8, 1.0], 'k--', alpha=0.5, label='x = y')

# Annotate drugs
for i, drug in enumerate(common_drugs):
    ax.annotate(drug, (baseline_aucs[i], esm2_aucs[i]),
                textcoords='offset points', xytext=(4, 4), fontsize=7)

# Legend for drug classes
for dc_label, c in color_map.items():
    ax.scatter([], [], c=c, s=60, label=dc_label)
ax.legend(loc='lower right', fontsize=10)

ax.set_xlabel('Baseline AUC (XGBoost + Mutations)', fontsize=13)
ax.set_ylabel('ESM-2 AUC (LR + Embeddings)', fontsize=13)
ax.set_title('B) Per-Drug AUC Comparison', fontsize=14, fontweight='bold')
ax.set_xlim([0.8, 1.0])
ax.set_ylim([0.8, 1.0])

# ── Panel C: Distribution of AUC improvements (histogram) ──
ax = axes[2]
improvements = [e - b for e, b in zip(esm2_aucs, baseline_aucs)]
ax.hist(improvements, bins=15, edgecolor='white', alpha=0.8, color='#9b59b6')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax.axvline(x=np.mean(improvements), color='red', linestyle='-', linewidth=2,
           label=f'Mean: {np.mean(improvements):+.4f}')
ax.set_xlabel('AUC Improvement (ESM-2 - Baseline)', fontsize=13)
ax.set_ylabel('Number of Drugs', fontsize=13)
ax.set_title('C) Distribution of AUC Improvements', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

n_better = sum(1 for imp in improvements if imp > 0)
ax.annotate(f'ESM-2 better: {n_better}/{len(improvements)} drugs',
            xy=(0.95, 0.95), xycoords='axes fraction',
            ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Figure 2: Statistical Comparison of Methods',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure2_method_comparison.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure2_method_comparison.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure2_method_comparison.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Classifier architecture comparison
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 3: Classifier comparison ===')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Panel A: Classifiers on frozen embeddings (horizontal bars) ──
ax = axes[0]
model_names = {
    'logistic': 'Logistic Regression',
    'xgboost': 'XGBoost',
    'rf': 'Random Forest',
}
model_means = []
model_stds = []
model_labels = []

for mt in ['logistic', 'xgboost', 'rf']:
    aucs = [r['auc'] for r in classifier_results[mt].values()]
    model_means.append(np.mean(aucs))
    model_stds.append(np.std(aucs))
    model_labels.append(model_names[mt])

y_pos = np.arange(len(model_labels))
bars = ax.barh(y_pos, model_means, xerr=model_stds, color=['#e74c3c', '#2ecc71', '#3498db'],
               alpha=0.85, edgecolor='white', capsize=5)
ax.set_yticks(y_pos)
ax.set_yticklabels(model_labels)
ax.set_xlabel('Mean AUC-ROC', fontsize=13)
ax.set_title('A) Classifiers on Frozen\nESM-2 Embeddings', fontsize=14, fontweight='bold')
ax.set_xlim([0.85, 1.0])
for bar, m, s in zip(bars, model_means, model_stds):
    ax.text(m + s + 0.003, bar.get_y() + bar.get_height()/2,
            f'{m:.4f}', va='center', fontsize=11, fontweight='bold')

# ── Panel B: Frozen vs fine-tuned comparison (grouped bars) ──
ax = axes[1]

# Get attention-model results by running per_drug_training with 'attention'
# We already have attention models trained; collect their AUCs via CV
# Use the logistic results as "frozen" and compare
frozen_aucs_by_class = {}
for dc in ['PI', 'NRTI', 'NNRTI']:
    aucs = []
    for drug in drug_lists[dc]:
        if drug in classifier_results['logistic']:
            aucs.append(classifier_results['logistic'][drug]['auc'])
    frozen_aucs_by_class[dc] = np.mean(aucs)

# Run attention CV for each class
print('Running attention-weighted CV for comparison...')
attn_aucs_by_class = {}
for dc in ['PI', 'NRTI', 'NNRTI']:
    print(f'  {dc}...')
    attn_res = per_drug_training(
        per_residue_embeddings[dc], pheno_dfs[dc], drug_lists[dc],
        model_type='attention', n_splits=5, random_state=42
    )
    attn_aucs = [r['auc'] for r in attn_res.values()]
    attn_aucs_by_class[dc] = np.mean(attn_aucs) if attn_aucs else 0

classes = ['PI', 'NRTI', 'NNRTI']
x = np.arange(len(classes))
width = 0.35

frozen_vals = [frozen_aucs_by_class[dc] for dc in classes]
attn_vals = [attn_aucs_by_class[dc] for dc in classes]

bars1 = ax.bar(x - width/2, frozen_vals, width, label='Frozen (LR)', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, attn_vals, width, label='Attention-Weighted', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel('Mean AUC-ROC', fontsize=13)
ax.set_title('B) Frozen vs Attention-Weighted\nPooling by Drug Class', fontsize=14, fontweight='bold')
ax.set_ylim([0.85, 1.0])
ax.legend(fontsize=11)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
            f'{h:.3f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Figure 3: Classifier Architecture Comparison',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure3_classifier_comparison.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure3_classifier_comparison.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure3_classifier_comparison.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: DRM enrichment validation
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 4: DRM enrichment ===')

fig = plot_drm_enrichment(validation_df, figsize=(20, 7),
                          save_path=str(FIGURES_DIR / 'figure4_drm_enrichment.png'))
fig.suptitle('Figure 4: DRM Enrichment Validation', fontsize=16, fontweight='bold', y=1.02)
plt.savefig(FIGURES_DIR / 'figure4_drm_enrichment.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure4_drm_enrichment.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 5: Attention differential profiles
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 5: Attention profiles ===')

fig, axes = plt.subplots(3, 1, figsize=(18, 18))

for idx, (drug, dc) in enumerate(representative_drugs.items()):
    if drug not in attention_differentials:
        print(f'  Skipping {drug}: no differential computed')
        continue

    diff_result = attention_differentials[drug]
    differential = diff_result['differential']
    drm_positions = get_drug_specific_drms(drug, dc)
    seq_len = len(differential)

    ax = axes[idx]
    positions = np.arange(1, seq_len + 1)

    # Color by direction
    colors = ['#e74c3c' if d > 0 else '#3498db' for d in differential]
    ax.bar(positions, differential, color=colors, width=1.0, edgecolor='none')

    # Highlight DRM positions
    for pos in drm_positions:
        if pos <= seq_len:
            ax.axvline(x=pos, color='green', alpha=0.3, linewidth=2, zorder=0)

    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xlabel('Sequence Position', fontsize=13)
    ax.set_ylabel('Attention Differential\n(Resistant - Susceptible)', fontsize=13)
    ax.set_title(f'{dc}/{drug}: ESM-2 Attention Profile\n(Green lines = known DRM positions)',
                 fontsize=14, fontweight='bold')
    ax.set_xlim(0, seq_len + 1)

    # Mark top 5 positions
    top5_idx = np.argsort(np.abs(differential))[-5:]
    for i in top5_idx:
        pos = i + 1
        val = differential[i]
        marker = '*' if pos in drm_positions else 'o'
        ax.annotate(f'{marker}{pos}', (pos, val),
                    textcoords='offset points',
                    xytext=(0, 5 if val > 0 else -10),
                    ha='center', fontsize=8, fontweight='bold')

fig.suptitle('Figure 5: Attention Differential Profiles for Representative Drugs',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure5_attention_profiles.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure5_attention_profiles.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure5_attention_profiles.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 6: Calibration curves (9-panel grid: 3 classes x 3 methods)
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 6: Calibration curves ===')

fig, axes = plt.subplots(3, 3, figsize=(18, 18))

method_names = ['Raw', 'Platt Scaling', 'Isotonic Regression']
method_keys = ['raw', 'platt', 'isotonic']

for row, dc in enumerate(['PI', 'NRTI', 'NNRTI']):
    cal = calibration_data[dc]
    y_true = cal['y_true']

    for col, (method_name, method_key) in enumerate(zip(method_names, method_keys)):
        ax = axes[row, col]
        y_pred = cal[method_key]

        # Compute calibration curve
        n_bins = 10
        bins = np.linspace(0, 1, n_bins + 1)
        bin_indices = np.digitize(y_pred, bins[1:-1])
        bin_means = []
        bin_true_probs = []
        bin_counts = []
        for i in range(n_bins):
            mask = bin_indices == i
            if mask.sum() > 0:
                bin_means.append(y_pred[mask].mean())
                bin_true_probs.append(y_true[mask].mean())
                bin_counts.append(mask.sum())

        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
        if bin_means:
            ax.bar(bin_means, bin_true_probs, width=0.08, alpha=0.7, color='#3498db', label='Model')
            ax.scatter(bin_means, bin_true_probs, s=[c/5 for c in bin_counts],
                       c='red', alpha=0.6, zorder=5)

        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])

        # Compute ECE
        if len(y_true) > 0:
            cal_metrics = compute_calibration_metrics(y_true, y_pred, n_bins=10)
            ece = cal_metrics['ece']
            ax.annotate(f'ECE={ece:.3f}', xy=(0.05, 0.92), xycoords='axes fraction',
                        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        if row == 0:
            ax.set_title(method_name, fontsize=14, fontweight='bold')
        if row == 2:
            ax.set_xlabel('Mean Predicted Probability', fontsize=12)
        if col == 0:
            ax.set_ylabel(f'{dc}\nFraction of Positives', fontsize=12)

        if row == 0 and col == 0:
            ax.legend(loc='lower right', fontsize=9)

fig.suptitle('Figure 6: Calibration Curves by Drug Class and Calibration Method',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure6_calibration_curves.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure6_calibration_curves.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure6_calibration_curves.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 7: PLM comparison (bar chart + box plot)
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 7: PLM comparison ===')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Panel A: Bar chart by drug
ax = axes[0]
piv = plm_df.pivot(index='drug', columns='plm', values='auc').sort_index()
piv.plot(kind='bar', ax=ax, width=0.8, edgecolor='white')
ax.set_ylabel('AUC-ROC', fontsize=13)
ax.set_title('A) PLM Comparison: AUC by Drug', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.0])
ax.legend(title='PLM', fontsize=10)
ax.tick_params(axis='x', rotation=45)

# Panel B: Box plot
ax = axes[1]
sns.boxplot(data=plm_df, x='plm', y='auc', ax=ax, palette='Set2')
sns.stripplot(data=plm_df, x='plm', y='auc', ax=ax, color='black', alpha=0.4, size=4)
ax.set_ylabel('AUC-ROC', fontsize=13)
ax.set_xlabel('')
ax.set_title('B) Distribution Across 18 Drugs', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.0])

for i, plm in enumerate(plm_df['plm'].unique()):
    m = plm_df[plm_df['plm'] == plm]['auc'].mean()
    ax.text(i, 0.81, f'mean={m:.3f}', ha='center', fontsize=9, style='italic')

fig.suptitle('Figure 7: Multi-PLM Performance Comparison',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure7_plm_comparison.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure7_plm_comparison.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure7_plm_comparison.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 8: PLM comparison heatmap
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 8: PLM heatmap ===')

fig, ax = plt.subplots(figsize=(10, 12))
drug_order = PI_DRUGS + NRTI_DRUGS + NNRTI_DRUGS
piv_hm = plm_df.pivot(index='drug', columns='plm', values='auc')
piv_hm = piv_hm.reindex([d for d in drug_order if d in piv_hm.index])

sns.heatmap(piv_hm, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.85, vmax=1.0, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'AUC-ROC'})

ax.set_title('Figure 8: Multi-PLM Performance Heatmap', fontsize=14, fontweight='bold')

# Draw drug class separators
n_pi = len([d for d in PI_DRUGS if d in piv_hm.index])
n_nrti = len([d for d in NRTI_DRUGS if d in piv_hm.index])
ax.axhline(y=n_pi, color='black', linewidth=2)
ax.axhline(y=n_pi + n_nrti, color='black', linewidth=2)

# Add class labels on right
ax.text(len(piv_hm.columns) + 0.3, n_pi / 2, 'PI',
        fontsize=12, fontweight='bold', va='center')
ax.text(len(piv_hm.columns) + 0.3, n_pi + n_nrti / 2, 'NRTI',
        fontsize=12, fontweight='bold', va='center')
ax.text(len(piv_hm.columns) + 0.3, n_pi + n_nrti + (len(piv_hm) - n_pi - n_nrti) / 2, 'NNRTI',
        fontsize=12, fontweight='bold', va='center')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure8_plm_heatmap.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure8_plm_heatmap.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure8_plm_heatmap.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 9: Subtype-stratified performance
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 9: Subtype-stratified performance ===')

fig, ax = plt.subplots(figsize=(12, 7))
plot_data = strat_df.dropna(subset=['auc'])

if not plot_data.empty:
    sns.boxplot(data=plot_data, x='subtype', y='auc', hue='drug_class',
                ax=ax, palette='Set2')
    sns.stripplot(data=plot_data, x='subtype', y='auc', hue='drug_class',
                  ax=ax, dodge=True, alpha=0.4, size=4, legend=False)

    ax.set_ylabel('AUC-ROC', fontsize=13)
    ax.set_xlabel('HIV-1 Subtype', fontsize=13)
    ax.set_title('Figure 9: ESM-2 Performance by HIV Subtype',
                 fontsize=14, fontweight='bold')
    ax.set_ylim([0.7, 1.05])
    ax.legend(title='Drug Class', fontsize=10)

    # Add sample counts
    subtypes = plot_data['subtype'].unique()
    for i, st in enumerate(sorted(subtypes)):
        n = plot_data[plot_data['subtype'] == st].shape[0]
        ax.text(i, 0.72, f'n={n}', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure9_subtype_stratified.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure9_subtype_stratified.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure9_subtype_stratified.png/pdf')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 10: Temporal holdout AUC
# ══════════════════════════════════════════════════════════════════════════════
print('=== Figure 10: Temporal holdout ===')

fig, ax = plt.subplots(figsize=(16, 7))

if not temp_df.empty:
    colors_map = {'PI': '#2196F3', 'NRTI': '#4CAF50', 'NNRTI': '#FF9800'}
    bar_colors = [colors_map.get(r['drug_class'], '#999') for _, r in temp_df.iterrows()]
    bars = ax.bar(range(len(temp_df)), temp_df['auc'], color=bar_colors, edgecolor='white')

    ax.set_xticks(range(len(temp_df)))
    ax.set_xticklabels(temp_df['drug'], rotation=45, ha='right')
    ax.set_ylabel('AUC-ROC', fontsize=13)
    ax.set_xlabel('Drug', fontsize=13)
    ax.set_title('Figure 10: Temporal Holdout Validation AUC',
                 fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.08)

    # Add value labels
    for bar, val in zip(bars, temp_df['auc']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=8)

    # Add mean line
    mean_auc = temp_df['auc'].mean()
    ax.axhline(y=mean_auc, color='red', linestyle='--', linewidth=1.5,
               label=f'Mean AUC = {mean_auc:.3f}')

    # Legend for drug classes
    for dc_label, c in colors_map.items():
        ax.bar([], [], color=c, label=dc_label)
    ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure10_temporal_holdout.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / 'figure10_temporal_holdout.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: figure10_temporal_holdout.png/pdf')

---
## Download all figures as a zip

In [ ]:
# ── Cell: Package and download all figures ───────────────────────────────
import zipfile

zip_path = 'manuscript_figures_400dpi.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(FIGURES_DIR.glob('figure*')):
        zf.write(f, f.name)
        print(f'  Added: {f.name}')

print(f'\nZip created: {zip_path}')

# List all generated figures
print('\n=== All generated figures ===')
for f in sorted(FIGURES_DIR.glob('figure*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}: {size_kb:.0f} KB')

from google.colab import files
files.download(zip_path)

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────
print('=' * 70)
print('FIGURE REGENERATION COMPLETE')
print('=' * 70)
print(f'\nAll figures saved at 400 DPI to: {FIGURES_DIR}/')
print(f'Both PNG and PDF versions generated.')
print(f'\nFigures:')
print(f'  1. Dataset composition (3 panels)')
print(f'  2. ESM-2 vs baseline comparison (3 panels)')
print(f'  3. Classifier architecture comparison (2 panels)')
print(f'  4. DRM enrichment validation (3 panels)')
print(f'  5. Attention differential profiles (3 panels)')
print(f'  6. Calibration curves (9-panel grid)')
print(f'  7. Multi-PLM comparison (2 panels)')
print(f'  8. PLM comparison heatmap')
print(f'  9. Subtype-stratified performance')
print(f' 10. Temporal holdout AUC')

# Key results summary
print(f'\n=== Key Results ===')
print(f'ESM-2 + LR mean AUC: {np.mean([r["auc"] for r in esm2_lr_results.values()]):.4f}')
print(f'Baseline mean AUC:   {np.mean([r["auc"] for r in baseline_results.values()]):.4f}')
if not temp_df.empty:
    print(f'Temporal holdout mean AUC: {temp_df["auc"].mean():.4f}')
print(f'\nPLM comparison:')
print(plm_df.groupby('plm')['auc'].agg(['mean', 'std']).round(4))